# Zain Jordan Customer 360 AI Workshop  
## Class 3: Natural Language SQL Agent for Telecom Business Questions

### Class Goal

In Class 1, we explored the Zain Jordan Customer 360 database using Python and SQL.  
In Class 2, we converted selected Python functions into LangChain tools and built a Customer Care AI Agent.  

In Class 3, we will build a **Natural Language SQL Agent**.

The SQL Agent can answer broader telecom business questions by:

1. Inspecting database tables.
2. Checking relevant table schemas.
3. Generating SQL.
4. Checking the SQL query.
5. Executing the query.
6. Explaining the result in business language.

---

## Main Class 3 Use Case

### Telecom Business Intelligence SQL Agent

Example questions:

- How many customers are there by city?
- Which cities have the most high-risk churn customers?
- Which customer segments generate the highest average revenue?
- What are the top complaint categories?
- Which campaigns have the highest conversion rate?

---

## Important Boundary

This class focuses on:

- LangChain SQL Agent
- Natural language to SQL
- Telecom business questions
- SQL safety and guardrails

We are **not** doing RAG, MCP, multi-agent, or external search in this class.


# 1. Learning Outcomes

By the end of this notebook, participants should be able to:

1. Understand the difference between a tools agent and a SQL agent.
2. Connect LangChain to the Zain Jordan SQLite database.
3. Create a SQLDatabase wrapper.
4. Create SQL tools using SQLDatabaseToolkit.
5. Build a SQL agent using `create_agent`.
6. Ask business questions in natural language.
7. Understand SQL agent risks and guardrails.
8. Use SQL Agent ideas for capstone projects.


# 2. Class 3 Concept: Tools Agent vs SQL Agent

| Class 2: Tools Agent | Class 3: SQL Agent |
|---|---|
| Uses predefined Python functions | Generates SQL dynamically |
| Best for controlled customer workflows | Best for flexible business questions |
| Example: Analyze customer 42 | Example: Which city has highest churn? |
| Safer and easier to control | More flexible but requires guardrails |
| Uses custom tools | Uses SQL database tools |

A SQL Agent is useful when a user wants to ask broad questions from the database without manually writing SQL.


# 3. Install Required Packages

Run this first in Google Colab.

We install:

- `langchain`
- `langchain-community`
- `langchain-openai`
- `langgraph`
- `pandas`
- `sqlalchemy`


In [ ]:
%pip install -q -U "langchain[openai]" langchain-community langchain-openai langgraph pandas sqlalchemy


# 4. Import Libraries


In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")


# 5. Set OpenAI API Key

In Google Colab:

1. Click the key icon on the left sidebar.
2. Add a secret named `OPENAI_API_KEY`.
3. Paste your OpenAI API key.
4. Enable notebook access for the secret.

This notebook will read the API key from Colab Secrets.


In [ ]:
try:
    from google.colab import userdata
    openai_key = userdata.get("OPENAI_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OpenAI API key loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")
except Exception:
    print("Not running in Google Colab, or Colab Secrets not available.")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. Agent cells will not run until it is configured.")
else:
    print("OPENAI_API_KEY is available.")


# 6. Upload or Locate the Zain Jordan Database

Upload the same database file used in the earlier classes:

`zain_customer_360_ai_demo.db`

If the file has a slightly different name, this notebook will automatically detect any `.db` file in the current folder.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available here. If running locally, place the .db file in the notebook folder.")


# 7. Locate Database File and Check Connection

This step uses plain SQLite first to confirm the database file is working.


In [ ]:
db_files = [file for file in os.listdir() if file.endswith(".db")]

if db_files:
    DB_PATH = db_files[0]
else:
    DB_PATH = "zain_customer_360_ai_demo.db"

db_path_obj = Path(DB_PATH).resolve()

print("Database path:", db_path_obj)
print("File exists:", db_path_obj.exists())

conn = sqlite3.connect(str(db_path_obj), check_same_thread=False)

tables_df = pd.read_sql_query("""
SELECT name 
FROM sqlite_master 
WHERE type = 'table'
ORDER BY name;
""", conn)

print("Number of tables:", len(tables_df))
tables_df


# 8. Quick Table Row Counts

This helps participants remember the telecom data landscape.


In [ ]:
table_counts = []

for table_name in tables_df["name"]:
    count_query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    count = pd.read_sql_query(count_query, conn)["row_count"][0]
    table_counts.append({
        "table_name": table_name,
        "row_count": count
    })

table_counts_df = pd.DataFrame(table_counts)
table_counts_df.sort_values("row_count", ascending=False)


# 9. Manual SQL Warm-Up

Before we let the SQL agent generate queries, we show learners what a normal SQL query looks like.

This also helps them compare the agent's answers later.


In [ ]:
manual_query = """
SELECT 
    city,
    COUNT(*) AS total_customers
FROM customers
GROUP BY city
ORDER BY total_customers DESC
LIMIT 10;
"""

pd.read_sql_query(manual_query, conn)


# 10. Manual Business Query: High Churn by City

This is a good business-intelligence question.

Question:

**Which cities have the most high-risk churn customers?**


In [ ]:
manual_query = """
SELECT 
    c.city,
    COUNT(*) AS high_risk_customers
FROM customers c
JOIN customer_churn_scores ch
    ON c.customer_id = ch.customer_id
WHERE ch.risk_level = 'High'
GROUP BY c.city
ORDER BY high_risk_customers DESC
LIMIT 10;
"""

pd.read_sql_query(manual_query, conn)


# 11. Manual Business Query: Revenue by Value Segment

Question:

**Which value segment has the highest average ARPU and revenue?**


In [ ]:
manual_query = """
SELECT 
    value_segment,
    COUNT(*) AS total_customers,
    ROUND(AVG(arpu_jod), 2) AS avg_arpu_jod,
    ROUND(AVG(total_revenue_6m_jod), 2) AS avg_revenue_6m_jod
FROM customer_value_segments
GROUP BY value_segment
ORDER BY avg_revenue_6m_jod DESC;
"""

pd.read_sql_query(manual_query, conn)


# 12. Create LangChain SQLDatabase Wrapper

LangChain uses `SQLDatabase` to connect to SQL databases and expose schema/query tools to the agent.

For SQLite, the URI format is:

`sqlite:////absolute/path/to/file.db`


In [ ]:
from langchain_community.utilities import SQLDatabase

db_uri = f"sqlite:///{db_path_obj}"
db = SQLDatabase.from_uri(db_uri)

print("Dialect:", db.dialect)
print("Usable tables:", db.get_usable_table_names())


# 13. Inspect One Table Through LangChain

This shows how LangChain can see table schema and sample data.


In [ ]:
print(db.get_table_info(["customers"]))


# 14. Create the Chat Model

We use LangChain's `init_chat_model`.

You can change the model name depending on what is available in your account.

Recommended starter model:

`gpt-4.1-mini`

If your account has a different model, update `MODEL_NAME`.


In [ ]:
from langchain.chat_models import init_chat_model

MODEL_NAME = "gpt-4.1-mini"

model = init_chat_model(
    MODEL_NAME,
    model_provider="openai",
    temperature=0
)

print("Model initialized:", MODEL_NAME)


# 15. Create SQL Database Tools

The SQL toolkit gives the agent tools such as:

- list tables
- inspect schema
- check SQL query
- execute SQL query

The agent will use these tools to answer questions.


In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=model)
sql_tools = toolkit.get_tools()

for tool in sql_tools:
    print("Tool:", tool.name)
    print("Description:", tool.description[:500])
    print("-" * 100)


# 16. Create the SQL Agent

The system prompt is very important.

We instruct the SQL agent to:

1. Start by checking tables.
2. Inspect relevant schemas.
3. Limit results.
4. Never use destructive SQL.
5. Double-check queries before execution.
6. Explain the answer in business language.


In [ ]:
from langchain.agents import create_agent

TOP_K = 5

system_prompt = f"""
You are a professional telecom business intelligence SQL agent for Zain Jordan.

You are connected to a SQLite telecom Customer 360 database.

Your job:
- Answer business questions using SQL.
- Inspect tables and schemas before writing SQL.
- Generate syntactically correct SQLite queries.
- Double-check SQL queries before execution.
- Execute queries only after checking them.
- Explain results in clear business language.

Important safety rules:
- Only use SELECT queries.
- Never use INSERT, UPDATE, DELETE, DROP, ALTER, TRUNCATE, or CREATE.
- Do not modify the database.
- Do not query all columns from a table unless absolutely necessary.
- Unless the user requests a specific number, limit query results to at most {TOP_K} rows.
- Use relevant columns only.
- If the question is ambiguous, explain your assumption.
- If a table or column does not exist, inspect schema and correct the query.
- Do not guess facts that are not in the database.

Useful telecom context:
- Churn analysis usually involves customers and customer_churn_scores.
- Revenue analysis usually involves customer_value_segments, invoices, payments, or transactions.
- Complaint analysis usually involves complaints and support_interactions.
- Campaign analysis usually involves campaigns and customer_campaign_responses.
- Network analysis usually involves network_towers and network_events.

After querying, provide:
1. Direct Answer
2. Key Numbers
3. Business Interpretation
4. Recommended Next Action, when useful
"""

sql_agent = create_agent(
    model=model,
    tools=sql_tools,
    system_prompt=system_prompt,
)

print("SQL Agent created successfully.")


# 17. Helper Function: Print Agent Steps

This function streams the agent output so learners can see tool calls and final answer.

This is useful for teaching because it shows how the agent thinks through tools:

- list tables
- inspect schema
- check SQL
- run SQL
- answer


In [ ]:
def run_sql_agent_verbose(question: str):
    print("USER QUESTION:")
    print(question)
    print("=" * 100)

    for step in sql_agent.stream(
        {"messages": [{"role": "user", "content": question}]},
        stream_mode="values",
    ):
        message = step["messages"][-1]
        try:
            message.pretty_print()
        except Exception:
            print(message)
        print("-" * 100)


# 18. Helper Function: Final Answer Only

Use this when you do not want to show all tool calls.


In [ ]:
def extract_final_text(result):
    last_message = result["messages"][-1]

    if hasattr(last_message, "content") and isinstance(last_message.content, str):
        return last_message.content

    if hasattr(last_message, "content_blocks"):
        parts = []
        for block in last_message.content_blocks:
            if isinstance(block, dict):
                if "text" in block:
                    parts.append(block["text"])
                elif "content" in block:
                    parts.append(str(block["content"]))
                else:
                    parts.append(str(block))
            else:
                parts.append(str(block))
        return "\\n".join(parts)

    return str(last_message)


def run_sql_agent(question: str):
    result = sql_agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })
    return extract_final_text(result)


# 19. Demo 1: Simple Count Question

Start with a very simple question.


In [ ]:
question = "How many customers are in the database?"

run_sql_agent_verbose(question)


# 20. Demo 2: Customers by City


In [ ]:
question = "Show the top 10 cities by number of customers."

answer = run_sql_agent(question)
print(answer)


# 21. Demo 3: Churn Risk Summary


In [ ]:
question = "How many customers are high, medium, and low churn risk?"

answer = run_sql_agent(question)
print(answer)


# 22. Demo 4: High Churn by City

This is one of the most useful telecom business questions.


In [ ]:
question = "Which cities have the most high-risk churn customers? Show the top 10 cities."

answer = run_sql_agent(question)
print(answer)


# 23. Demo 5: Revenue by Value Segment


In [ ]:
question = "Which customer value segments have the highest average ARPU and six-month revenue?"

answer = run_sql_agent(question)
print(answer)


# 24. Demo 6: High Value + High Churn Risk

This question is directly useful for retention teams.


In [ ]:
question = """
Find the top 10 customers who are high churn risk and also high value.
Include customer ID, name, city, value segment, ARPU, six-month revenue, churn score, and main risk reason.
"""

answer = run_sql_agent(question)
print(answer)


# 25. Demo 7: Complaint Analysis


In [ ]:
question = "What are the top complaint categories by volume and severity?"

answer = run_sql_agent(question)
print(answer)


# 26. Demo 8: Support Channel Analysis


In [ ]:
question = """
Which support channels have the highest number of negative customer sentiment interactions?
Show the channel and count.
"""

answer = run_sql_agent(question)
print(answer)


# 27. Demo 9: Campaign Performance


In [ ]:
question = """
Which campaigns had the highest conversion rate?
Show campaign name, campaign type, target segment, total sent, total converted, and conversion rate.
"""

answer = run_sql_agent(question)
print(answer)


# 28. Demo 10: Network Events


In [ ]:
question = """
Which cities had network events affecting the largest number of customers?
Show city, technology, event type, severity, total events, and total affected customers.
"""

answer = run_sql_agent(question)
print(answer)


# 29. Business Insight Challenge

Now ask the agent to behave like a business analyst.

This is useful for capstone preparation.


In [ ]:
question = """
Act as a telecom business analyst.

Find 3 useful insights from the Zain Jordan Customer 360 database.
For each insight:
1. Mention the business question you answered
2. Mention the key numbers
3. Explain the business meaning
4. Recommend one action
"""

answer = run_sql_agent(question)
print(answer)


# 30. Exercise 1: Simple Business Questions

Ask the SQL Agent:

1. How many customers are there?
2. How many plans are available?
3. How many complaints are in the database?
4. How many network events are recorded?


In [ ]:
exercise_question = "How many plans are available in the database?"

answer = run_sql_agent(exercise_question)
print(answer)


# 31. Exercise 2: Grouping Questions

Try these:

1. How many customers are there by city?
2. How many customers are there by customer segment?
3. How many complaints are there by severity?
4. How many support interactions are there by channel?


In [ ]:
exercise_question = "How many complaints are there by severity?"

answer = run_sql_agent(exercise_question)
print(answer)


# 32. Exercise 3: Business Insight Questions

Try these:

1. Which cities have the highest churn risk?
2. Which customer value segments have the highest ARPU?
3. Which complaint categories are most common?
4. Which campaigns have the best conversion rate?
5. Which support channel has the most negative sentiment?


In [ ]:
exercise_question = "Which support channel has the most negative sentiment?"

answer = run_sql_agent(exercise_question)
print(answer)


# 33. Exercise 4: Capstone Thinking

Ask:

> What business problem can we solve using this SQL Agent?

Possible projects:

- Churn analytics assistant
- Revenue insight assistant
- Customer complaint dashboard assistant
- Campaign performance assistant
- Executive KPI assistant
- Network issue analysis assistant


In [ ]:
capstone_question = """
Suggest 5 capstone project ideas that can use this SQL agent and the Zain Jordan customer database.
For each idea, mention the target user, key questions, and expected output.
"""

answer = run_sql_agent(capstone_question)
print(answer)


# 34. SQL Agent Safety and Guardrails

SQL agents are powerful but need careful controls.

## For Training

We are using:

- Synthetic data
- Local SQLite database
- Read-only style prompts
- Result limits

## For Production

You should use:

1. Read-only database credentials.
2. Table-level permissions.
3. Column restrictions for sensitive data.
4. Query allowlist or validation.
5. Row limits.
6. Logging and monitoring.
7. Human review for sensitive insights.
8. No direct access to production customer PII unless approved.

## Important Rule

Never give an LLM unrestricted access to a sensitive production database.


# 35. Mistakes to Avoid

## Mistake 1: Asking vague questions

Bad:

> Tell me about customers.

Better:

> Show the top 10 cities by number of customers and explain what it means.

---

## Mistake 2: Asking too much at once

Bad:

> Analyze everything.

Better:

> Find the top 5 complaint categories by volume and severity.

---

## Mistake 3: Trusting the result blindly

Always check:

- Which tables were used?
- Does the result make business sense?
- Was the query too broad?
- Are filters missing?
- Is the row limit appropriate?

---

## Mistake 4: No business interpretation

SQL results are not enough.

A good AI system should explain:

- What happened
- Why it matters
- What action to take


# 36. What We Built Today

In Class 3, we built:

1. A LangChain SQLDatabase connection to the Zain Jordan SQLite database.
2. SQL tools using SQLDatabaseToolkit.
3. A natural language SQL Agent using create_agent.
4. Business question demos.
5. Churn, revenue, complaint, campaign, support, and network analysis examples.
6. Capstone project prompts.
7. SQL safety and guardrail discussion.

---

## Next Class

In Class 4, we will build a **RAG Assistant from Telecom Database Content**.

Instead of only generating SQL, we will convert selected database rows into searchable documents and retrieve relevant context for plan, campaign, support, and customer-care recommendations.


# 37. Trainer Closing Script

Today, we moved from controlled tools to flexible database analytics.

In Class 2, our agent used predefined tools.  
In Class 3, our SQL agent could inspect the database, generate SQL, execute the query, and explain business results.

This is useful for:

- Churn analysis
- Campaign analysis
- Complaint intelligence
- Revenue insights
- Executive dashboards
- Network impact analysis

In the next class, we will move to RAG, where we create a searchable knowledge layer from selected telecom data.
